In [2]:
import os
import json
import ollama

# 1. ตั้งค่า Path ของไฟล์ Text ที่บันทึกไว้
txt_folder = '/Users/faiijaran/Desktop/project/version_2_photo_with_png/output/extracted_text'

def ask_llama_v2(text_content):
    """ส่งข้อความที่เรียงบรรทัดแล้วให้ Llama 3.2 ช่วยสรุปผลเป็น JSON"""
    prompt = f"""
    คุณคือผู้เชี่ยวชาญด้านบัญชีไทย หน้าที่ของคุณคือสกัดข้อมูลจาก OCR ใบเสร็จ/ใบเสนอราคา ให้เป็น JSON ที่ถูกต้องที่สุด
    
    กฎการประมวลผล:
    1. ซ่อมแซมตัวเลขที่ OCR อ่านผิด: หากอยู่ในช่องจำนวนเงินหรือจำนวนสินค้า ให้เปลี่ยน 'o' หรือ 'O' เป็น '0' (เลขศูนย์) เสมอ
    2. จัดการวันที่: หากปีเป็น พ.ศ. (25xx) ให้ลบ 543 เพื่อเปลี่ยนเป็น ค.ศ. (20xx) เช่น '4 สิงหาคม 2025' หรือ '2568' -> '2025'
    3. สกัดข้อมูลเป็น JSON ที่มีโครงสร้างดังนี้:
       - company: ชื่อบริษัทผู้ออกเอกสาร
       - document_type: ประเภทเอกสาร (เช่น Receipt, Quotation, Invoice)
       - date: วันที่ (รูปแบบ YYYY-MM-DD)
       - total: ยอดรวมสุทธิ (Float)
       - items: รายการสินค้า (List ของ {{ "name": ..., "qty": ..., "price": ... }})

    ข้อความ OCR:
    {text_content}
    """
    
    # เรียกใช้ Llama 3.2 ผ่าน Ollama
    response = ollama.chat(
        model='llama3.2',
        messages=[{'role': 'user', 'content': prompt}],
        format='json' # บังคับให้ AI ตอบกลับเป็นโครงสร้าง JSON
    )
    return response['message']['content']

# 2. เริ่มวิเคราะห์ไฟล์ 10 ไฟล์
print("🤖 Llama 3.2 กำลังวิเคราะห์ข้อมูล Version 2 (Digital Documents)...")
print("=" * 60)

# ดึงไฟล์ text_1.txt ถึง text_10.txt
files = [f for f in os.listdir(txt_folder) if f.startswith('text_') and f.endswith('.txt')]
files.sort(key=lambda x: int(''.join(filter(str.isdigit, x))))

for filename in files:
    file_path = os.path.join(txt_folder, filename)
    
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            raw_text = f.read()
        
        # ส่งให้ AI วิเคราะห์
        json_result = ask_llama_v2(raw_text)
        data = json.loads(json_result)
        
        # --- แสดงผลที่หน้าจอ Jupyter ---
        print(f"📍 [วิเคราะห์เสร็จสิ้น: {filename}]")
        print(f"🏢 ร้าน: {data.get('company')}")
        print(f"📅 วันที่: {data.get('date')} | 💰 ยอดรวม: {data.get('total')}")
        print(f"📝 ประเภท: {data.get('document_type')}")
        print("-" * 30)
        print("📦 รายการสินค้า:")
        for item in data.get('items', []):
            print(f"  - {item.get('name')} x{item.get('qty')} @{item.get('price')}")
        print("=" * 60)
        
    except Exception as e:
        print(f"❌ พลาดที่ไฟล์ {filename}: {e}")
        print("=" * 60)

print("\n✨ ตรวจสอบผลลัพธ์ JSON ในคอนโซลด้านบนได้เลยครับ!")

🤖 Llama 3.2 กำลังวิเคราะห์ข้อมูล Version 2 (Digital Documents)...
📍 [วิเคราะห์เสร็จสิ้น: text_1.txt]
🏢 ร้าน: cony
📅 วันที่: 2563-01-23 | 💰 ยอดรวม: 51260
📝 ประเภท: receipt
------------------------------
📦 รายการสินค้า:
  - าบย์ัอ็ มค่ x1 @10240
  - None x2 @20480
  - None x2 @30720
  - None x3 @48000
  - None x2 @10200
  - None x4 @57600
  - None x5 @102000
  - None x5 @204000
  - None x8 @255200
📍 [วิเคราะห์เสร็จสิ้น: text_2.txt]
🏢 ร้าน: บริษัท ดาต้าดีไซน์ สตูดิโอ จำกัด
📅 วันที่: 2568-04-08 | 💰 ยอดรวม: 3000.0
📝 ประเภท: ใบเสร็จ
------------------------------
📦 รายการสินค้า:
  - ข์สนปัรตยถล x20 @300.0
  - ข์สนปัรตยถล x3 @300.0
📍 [วิเคราะห์เสร็จสิ้น: text_3.txt]
🏢 ร้าน: บริษัท บีมเอเจนซี่ แอนด์ ดิจิทอล จำกัด
📅 วันที่: 2025-08-04 | 💰 ยอดรวม: 9630.0
📝 ประเภท: ใบเสร็จ
------------------------------
📦 รายการสินค้า:
  - ผวษ ส๋หตาถเ x5 @5000.0
  - None x2 @2000.0
📍 [วิเคราะห์เสร็จสิ้น: text_4.txt]
🏢 ร้าน: จรัญญา บุญทอง
📅 วันที่: 2025-08-04 | 💰 ยอดรวม: 310.0
📝 ประเภท: ใบเสนอราคา
----------------